# Evaluation IBM Dataset - Experiment 3 (LLM Context Selection)

Evaluates the prompts generated by `Creating IBM Dataset Prompts (Experiment 3).ipynb`, in which an LLM selects the N most relevant terms from the argument and topic and prunes the K candidates retrieved per term from the Google Knowledge Graph, instead of naively searching every POS-tagged term as in Experiment 2.

This notebook evaluates a single parameterized combination of prompts file (`N_TERMS`, `K_DEPTH`, `PROMPT_MODEL_TYPE`) using a chosen evaluation model (`EVAL_MODEL_TYPE`, independent from the model that generated the prompts). It is otherwise identical to the Experiment 3 evaluation already used elsewhere in this project, minus the sentiment-analysis step.

A summary section at the end reports which of the 8 `(N, K, model)` result files are still missing for this dataset, and aggregates the metrics of every result file already created.

In [ ]:
!pip install -q -U google-generativeai pandas scikit-learn tqdm

In [ ]:
import os
import itertools
import pandas as pd
import google.generativeai as genai
from sklearn.metrics import accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import drive

In [ ]:
# --- API and Drive Configuration ---
try:
    GOOGLE_API_KEY = ''  # Your Gemini API key here
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Gemini API configured.")
except Exception as e:
    print(f"API key not found or invalid. Error: {e}")

try:
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except Exception:
    print("Google Drive already mounted or error mounting.")

## Parameters

Choose which Experiment 3 prompts file to evaluate (`N_TERMS`, `K_DEPTH`, `PROMPT_MODEL_TYPE`) and which model should actually answer ATTACK/SUPPORT (`EVAL_MODEL_TYPE`). The two choices are independent: by default they can match, but you can evaluate a prompts file generated with one model using the other model.

In [ ]:
DATASET = "IBM"  # fixed for this notebook
BASE_DIR = '/content/drive/MyDrive/ProjectFolder'

# --- Experiment 3 combination grid (8 combinations per dataset: N x K x model) ---
N_OPTIONS = [1, 5]
K_OPTIONS = [1, 5]
MODEL_TYPE_OPTIONS = [31, 38]
MODEL_NAME_MAP = {
    31: 'gemini-3.1-flash-lite',
    38: 'gemini-3.8-flash',
}

# --- Pick which prompts file to load for THIS run ---
N_TERMS = 5               # one of N_OPTIONS
K_DEPTH = 5               # one of K_OPTIONS
PROMPT_MODEL_TYPE = 31    # one of MODEL_TYPE_OPTIONS - the model that generated/pruned this prompt's context

# --- Pick which model will actually be called to answer ATTACK/SUPPORT ---
EVAL_MODEL_TYPE = 31      # one of MODEL_TYPE_OPTIONS - independent of PROMPT_MODEL_TYPE above

assert N_TERMS in N_OPTIONS, f"N_TERMS must be one of {N_OPTIONS}"
assert K_DEPTH in K_OPTIONS, f"K_DEPTH must be one of {K_OPTIONS}"
assert PROMPT_MODEL_TYPE in MODEL_TYPE_OPTIONS, f"PROMPT_MODEL_TYPE must be one of {MODEL_TYPE_OPTIONS}"
assert EVAL_MODEL_TYPE in MODEL_TYPE_OPTIONS, f"EVAL_MODEL_TYPE must be one of {MODEL_TYPE_OPTIONS}"


def prompts_path(dataset, n, k, model_type):
    return f'{BASE_DIR}/prompts_{dataset}_{n}_{k}_{model_type}_experiment3.csv'


def results_path(dataset, n, k, model_type):
    return f'{BASE_DIR}/results_{dataset}_{n}_{k}_{model_type}_experiment3.csv'


EXPERIMENT3_PROMPTS_PATH = prompts_path(DATASET, N_TERMS, K_DEPTH, PROMPT_MODEL_TYPE)
EXPERIMENT3_RESULTS_PATH = results_path(DATASET, N_TERMS, K_DEPTH, PROMPT_MODEL_TYPE)
eval_model = genai.GenerativeModel(MODEL_NAME_MAP[EVAL_MODEL_TYPE])

print(f"Prompts file: {EXPERIMENT3_PROMPTS_PATH}")
print(f"Results file: {EXPERIMENT3_RESULTS_PATH}")
print(f"Evaluation model: {MODEL_NAME_MAP[EVAL_MODEL_TYPE]} (EVAL_MODEL_TYPE={EVAL_MODEL_TYPE})")

## Step 1: Load the Experiment 3 prompts

In [ ]:
try:
    df_exp3 = pd.read_csv(EXPERIMENT3_PROMPTS_PATH)
    print(f"File '{EXPERIMENT3_PROMPTS_PATH}' loaded successfully. Total of {len(df_exp3)} rows.")
    display(df_exp3.head())
except FileNotFoundError:
    print(f"ERROR: File '{EXPERIMENT3_PROMPTS_PATH}' not found. Make sure the Experiment 3 "
          f"prompt-creation notebook was run first for N_TERMS={N_TERMS}, K_DEPTH={K_DEPTH}, "
          f"PROMPT_MODEL_TYPE={PROMPT_MODEL_TYPE}.")
    df_exp3 = None

In [ ]:
def _call_gemini_api_with_retry(prompt, model):
    """Helper function that calls the API with retry logic.
    The prompt must already contain the full instruction."""
    for attempt in range(2):
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if "ATTACK" in answer:
                return "ATTACK"
            elif "SUPPORT" in answer:
                return "SUPPORT"
            else:
                return "N/A"
        except Exception as e:
            if attempt == 1:
                print(f"Final error processing prompt: {str(e)}")
                return "ERROR"
    return "ERROR"

## Step 2: Single-case test

In [ ]:
if df_exp3 is not None:
    random_row = df_exp3.sample(1).iloc[0]
    answer_exp3 = _call_gemini_api_with_retry(random_row['final_prompt_llm'], eval_model)

    print("\n[PROMPT: EXPERIMENT 3 - LLM CONTEXT SELECTION]")
    print(random_row['final_prompt_llm'])
    print("-" * 20)
    print(f"Ground truth: {random_row['ground_truth_stance']}")
    print(f"Model answer: {answer_exp3}")

## Step 3: Parallel batch evaluation

In [ ]:
if df_exp3 is not None:
    print("\n--- Starting batch evaluation for Experiment 3 ---")

    results_exp3 = [None] * len(df_exp3)

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(_call_gemini_api_with_retry, row['final_prompt_llm'], eval_model): index
            for index, row in df_exp3.iterrows()
        }

        for future in tqdm(as_completed(futures), total=len(df_exp3), desc="Evaluating Experiment 3 prompts"):
            original_index = futures[future]
            try:
                results_exp3[original_index] = future.result()
            except Exception as e:
                print(f"Error in thread for index {original_index}: {e}")
                results_exp3[original_index] = "ERROR"

    df_exp3['gemini_experiment3'] = results_exp3
    df_exp3['eval_model_type'] = EVAL_MODEL_TYPE
    df_exp3['eval_model_name'] = MODEL_NAME_MAP[EVAL_MODEL_TYPE]

    df_exp3.to_csv(EXPERIMENT3_RESULTS_PATH, index=False, encoding='utf-8')
    print(f"\n--- Evaluation completed. Results saved to '{EXPERIMENT3_RESULTS_PATH}' ---")

## Step 4: Metrics

In [ ]:
if df_exp3 is not None and 'gemini_experiment3' in df_exp3.columns:
    y_true_exp3 = df_exp3['ground_truth_stance']
    y_pred_exp3 = df_exp3['gemini_experiment3']
    valid_exp3 = df_exp3[df_exp3['gemini_experiment3'].isin(['SUPPORT', 'ATTACK'])]

    acc_exp3 = accuracy_score(y_true_exp3, y_pred_exp3)
    f1_exp3 = f1_score(valid_exp3['ground_truth_stance'], valid_exp3['gemini_experiment3'], average='weighted')

    print(f"\n[Results for: N={N_TERMS}, K={K_DEPTH}, prompt_model={MODEL_NAME_MAP[PROMPT_MODEL_TYPE]}, eval_model={MODEL_NAME_MAP[EVAL_MODEL_TYPE]}]")
    print(f"Accuracy: {acc_exp3:.4f} (including errors)")
    print(f"F1-Score (Weighted): {f1_exp3:.4f} (valid answers only)")
else:
    print("\nExperiment 3 evaluation has not been run yet.")

## Summary: missing result files and aggregated metrics

(i) lists, out of the 8 `(N, K, model)` combinations expected for the **IBM** dataset, which result files have not been created yet.
(ii) loads every result file that already exists and repeats the metric computation above for all of them, so you get a joint view of everything evaluated so far.

In [ ]:
print("\n" + "=" * 80)
print(f" SUMMARY - EXPERIMENT 3 ({DATASET})")
print("=" * 80)

combos = list(itertools.product(N_OPTIONS, K_OPTIONS, MODEL_TYPE_OPTIONS))

print(f"\n--- (i) Missing result files (out of {len(combos)} N x K x model combinations) ---")
missing_combos = []
existing_combos = []
for n, k, mt in combos:
    rpath = results_path(DATASET, n, k, mt)
    if os.path.exists(rpath):
        existing_combos.append((n, k, mt, rpath))
    else:
        missing_combos.append((n, k, mt, rpath))

if missing_combos:
    for n, k, mt, rpath in missing_combos:
        print(f"  MISSING: N={n}, K={k}, model={MODEL_NAME_MAP[mt]} ({mt}) -> {os.path.basename(rpath)}")
else:
    print("  All 8 combinations already have a result file.")

print(f"\n--- (ii) Aggregated metrics for every combination already evaluated ({len(existing_combos)}/{len(combos)}) ---")

summary_rows = []
for n, k, mt, rpath in existing_combos:
    try:
        res_df = pd.read_csv(rpath)
    except Exception as e:
        print(f"Error loading '{rpath}': {e}")
        continue

    if 'gemini_experiment3' not in res_df.columns:
        print(f"N={n}, K={k}, model={mt}: column 'gemini_experiment3' not found in '{rpath}'.")
        continue

    y_true = res_df['ground_truth_stance'].astype(str)
    y_pred = res_df['gemini_experiment3'].astype(str)
    labels = sorted(y_true.dropna().unique())
    valid_mask = y_pred.isin(labels)

    if valid_mask.sum() == 0:
        print(f"N={n}, K={k}, model={mt}: no valid answers found.")
        continue

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average='weighted', labels=labels, zero_division=0)
    eval_model_used = res_df['eval_model_name'].iloc[0] if 'eval_model_name' in res_df.columns else 'unknown'

    summary_rows.append({
        'N': n,
        'K': k,
        'prompt_model': MODEL_NAME_MAP[mt],
        'eval_model': eval_model_used,
        'n_rows': len(res_df),
        'accuracy': acc,
        'f1_weighted': f1,
    })

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    display(summary_df.sort_values(['N', 'K', 'prompt_model']).reset_index(drop=True))
else:
    print("  No result file could be evaluated yet.")